# Edit Wikidata using wikibaseintegrator

In [39]:
from wikibaseintegrator import WikibaseIntegrator
from wikibaseintegrator import wbi_login
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator import datatypes
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)

BOT_AUTH = authorization['BOT_AUTH']
USERNAME = authorization['BOT_USERNAME']
PASSWORD = authorization['BOT_USER_PWD']
#CONSUMER_TOKEN = authorization['consumer_token']
#CONSUMER_SECRET = authorization['consumer_secret']

BOTNAME = authorization['BOTNAME']
    

log_file = str(Path('logs/wikibaseint-debug.log'))
logging.basicConfig(filename=log_file, force=True,
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{USERNAME})'
#wbi_config['MEDIAWIKI_API_URL'] = 'https://test.wikidata.org/w/api.php'
wbi_config['MEDIAWIKI_API_URL'] = 'https://wikidata.org/w/api.php'


PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'language':'P407',
         'publication_date':'P577'}
ENTITIES = {
   'literary_work':'Q7725634', 
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

### Getting an item without logging in

To edit, you will need to log in

In [40]:
wbi = WikibaseIntegrator()
gefaegnis_pgde = wbi.item.get(entity_id='Q136794803', mediawiki_api_url='https://www.wikidata.org/w/api.php') 
gefaegnis_pgde


<ItemEntity @bb6db0 _BaseEntity__api=<wikibaseintegrator.wikibaseintegrator.WikibaseIntegrator object at 0x10b565dc0>
	 _BaseEntity__title='Q136794803'
	 _BaseEntity__pageid=130475469
	 _BaseEntity__lastrevid=2451603218
	 _BaseEntity__type='item'
	 _BaseEntity__id='Q136794803'
	 _BaseEntity__claims=<Claims @567b60 _Claims__claims={'P31': [<Item @567aa0 _Claim__mainsnak=<Snak @566510 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P31' _Snak__hash='7d261aaf92dd9be13f2bd79d7024b98d449f1f89' _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 3331189, 'id': 'Q3331189'}, 'type': 'wikibase-entityid'} _Snak__datatype='wikibase-item'> _Claim__type='statement' _Claim__qualifiers=<Qualifiers @567020 _Qualifiers__qualifiers={}> _Claim__qualifiers_order=[] _Claim__id='Q136794803$97FFA24A-0320-430C-A422-481EAF8AC226' _Claim__rank=<WikibaseRank.NORMAL: 'normal'> _Claim__removed=False _Claim__references=<References @567cb0 _References__references=[]>>], '

### Logging in using the old method

In [ ]:
#login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD)
login_instance = wbi_login.Login(user=BOT_AUTH, password=PASSWORD, mediawiki_api_url='https://www.wikidata.org/w/api.php')

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

In [42]:
wbi = WikibaseIntegrator()
gefaegnis_pgde = wbi.item.get(entity_id='Q136794803', mediawiki_api_url='https://www.wikidata.org/w/api.php') 

### Logging in using OAuth (preferred)

In [ ]:
oauth = wbi_login.OAuth2(consumer_token=CONSUMER_TOKEN, 
                                  consumer_secret=CONSUMER_SECRET)
wbi_oauth = WikibaseIntegrator(login=oauth)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'


Get this entity to test:  https://www.wikidata.org/wiki/Q105624761

## Add a claim

What titles are there now?

In [ ]:
flametti = wbi.item.get(entity_id='Q105624761')

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

Now we add this title to our local python representation of this entity

In [ ]:
title_en_string = datatypes.MonolingualText(text='Flametti, or The Dandyism of the Poor', language='en', prop_nr=PROPS['title'])
title_en_string

flametti.claims.add(title_en_string)

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

## Write it to the instance

In [ ]:
flametti = flametti.write(login=login_instance)

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

See:  https://www.wikidata.org/wiki/Q105624761

### With previous versions of WikibaseIntegrator, one had to do it like this:

In [ ]:
flametti.claims.add(title_en_string, action_if_exists=ActionIfExists.KEEP)

## Deleting a claim

In [26]:
item = wbi.item.get(entity_id='Q136794803', mediawiki_api_url='https://www.wikidata.org/w/api.php') 
print(item)
print(f'nr of claims to delete: {len(item.claims.get("P144"))}')

<ItemEntity @bb4a70 _BaseEntity__api=<wikibaseintegrator.wikibaseintegrator.WikibaseIntegrator object at 0x109bb4e00>
	 _BaseEntity__title='Q136794803'
	 _BaseEntity__pageid=130475469
	 _BaseEntity__lastrevid=2451602838
	 _BaseEntity__type='item'
	 _BaseEntity__id='Q136794803'
	 _BaseEntity__claims=<Claims @ae42c0 _Claims__claims={'P31': [<Item @e048f0 _Claim__mainsnak=<Snak @e065d0 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P31' _Snak__hash='7d261aaf92dd9be13f2bd79d7024b98d449f1f89' _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 3331189, 'id': 'Q3331189'}, 'type': 'wikibase-entityid'} _Snak__datatype='wikibase-item'> _Claim__type='statement' _Claim__qualifiers=<Qualifiers @e04f50 _Qualifiers__qualifiers={}> _Claim__qualifiers_order=[] _Claim__id='Q136794803$97FFA24A-0320-430C-A422-481EAF8AC226' _Claim__rank=<WikibaseRank.NORMAL: 'normal'> _Claim__removed=False _Claim__references=<References @e05bb0 _References__references=[]>>], '

In [27]:
for claim in item.claims.get('P144'):
    claim.remove()

item.write(mediawiki_api_url='https://www.wikidata.org/w/api.php')

item = wbi.item.get(entity_id='Q136794803', mediawiki_api_url='https://www.wikidata.org/w/api.php') 

print(f'nr of claims after writing: {len(item.claims.get("P144"))}')

nr of claims after writing: 0
